In [1]:
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
btns = base.btns_gpio

In [2]:
%%microblaze base.PMODA

// Messing around with the passive buzzer to do different tones


#include "gpio.h"
#include "timer.h"

// plugged my buzzer in upside down on ground pin so data pin D3 will activate the buzzer
const unsigned DPIN=2;
gpio d_pin;

// set pin DPIN to write and the other pins to read
int init(){
    d_pin = gpio_open(DPIN);
    gpio_set_direction(d_pin, GPIO_OUT);
    
    // turn off other pins
    gpio parent = gpio_open_device(0);
    gpio in_pins = gpio_configure(parent, 1, 3, 1);
    gpio_set_direction(in_pins, GPIO_OUT);
    gpio_write(in_pins,0);
    
    return 0;
}

unsigned int write(unsigned int signal){
    if(!d_pin) init();
    gpio_write(d_pin, signal);
    return signal;
}

int pwm(unsigned int freq, unsigned int duty, unsigned int val){
    if(!d_pin)init();
    
    freq = freq==0 ? 1:freq;
    duty = duty%101;
    
    unsigned int us = 1000000; // work in micro secs (us)
    unsigned int T =  (us / freq);

    unsigned int on_us  = T * duty / 100; // convert duty into percent
    unsigned int off_us = T - on_us;

    gpio_write(d_pin, val);
    delay_us(on_us); 
    gpio_write(d_pin, 0);
    delay_us(off_us);

    return 0;
}

unsigned int read(){
    if(!d_pin)init();
    return gpio_read(d_pin);
}

int tone(int f){
    for (unsigned int i = 0; i < f; i++) {
        pwm(f, 50, 1);
    }
    return 0;
}

int tone_ms(unsigned int f, unsigned int ms)
{
    // Rest (silence)
    if (f == 0) {
        write(0);
        // use pwm at 1Hz to approximate delay in chunks
        // 1Hz = 1 second per cycle
        unsigned int chunks = ms / 1000u;
        for (unsigned int i = 0; i < chunks; i++) {
            pwm(1, 0, 0);   // just burns time
        }
        return 0;
    }

    // number of PWM cycles needed
    unsigned int cycles = (f * ms) / 1000u;
    if (cycles == 0) cycles = 1;

    for (unsigned int i = 0; i < cycles; i++) {
        pwm(f, 90, 1);
    }

    write(0);
    return 0;
}
int twinkle()
{
    // Phrase 1: C C G G A A G
    tone_ms(2620, 400);
    tone_ms(2620, 400);
    tone_ms(3920, 400);
    tone_ms(3920, 400);
    tone_ms(4400, 400);
    tone_ms(4400, 400);
    tone_ms(3920, 800);

    tone_ms(0, 200);

    // Phrase 2: F F E E D D C
    tone_ms(3490, 400);
    tone_ms(3490, 400);
    tone_ms(3300, 400);
    tone_ms(3300, 400);
    tone_ms(2940, 400);
    tone_ms(2940, 400);
    tone_ms(2620, 800);

    tone_ms(0, 200);

    // Phrase 3: G G F F E E D
    tone_ms(3920, 400);
    tone_ms(3920, 400);
    tone_ms(3490, 400);
    tone_ms(3490, 400);
    tone_ms(3300, 400);
    tone_ms(3300, 400);
    tone_ms(2940, 800);

    tone_ms(0, 200);

    // Phrase 4: G G F F E E D
    tone_ms(3920, 400);
    tone_ms(3920, 400);
    tone_ms(3490, 400);
    tone_ms(3490, 400);
    tone_ms(3300, 400);
    tone_ms(3300, 400);
    tone_ms(2940, 800);

    tone_ms(0, 200);

    // Phrase 5: C C G G A A G
    tone_ms(2620, 400);
    tone_ms(2620, 400);
    tone_ms(3920, 400);
    tone_ms(3920, 400);
    tone_ms(4400, 400);
    tone_ms(4400, 400);
    tone_ms(3920, 800);

    tone_ms(0, 200);

    // Phrase 6: F F E E D D C
    tone_ms(3490, 400);
    tone_ms(3490, 400);
    tone_ms(3300, 400);
    tone_ms(3300, 400);
    tone_ms(2940, 400);
    tone_ms(2940, 400);
    tone_ms(2620, 800);

    return 0;
}

In [25]:
tone_ms(1500,2000)

0

In [17]:
twinkle()

0

In [68]:
tic = time.perf_counter()
tone(1000)
toc  = time.perf_counter()
print("NO FN Elapsed time:", toc - tic, "seconds")

NO FN Elapsed time: 1.0216711309994935 seconds


In [3]:
%%microblaze base.PMODB

#include "gpio.h"
#include "timer.h"

// plugged my buzzer in upside down on ground pin so data pin D3 will activate the buzzer
const unsigned DPIN=2;
gpio d_pin;

// set pin DPIN to write and the other pins to read
int init(){
    d_pin = gpio_open(DPIN);
    gpio_set_direction(d_pin, GPIO_OUT);
    
    // turn off other pins
    gpio parent = gpio_open_device(0);
    gpio in_pins = gpio_configure(parent, 1, 3, 1);
    gpio_set_direction(in_pins, GPIO_OUT);
    gpio_write(in_pins,0);
    
    return 0;
}

unsigned int write(unsigned int signal){
    if(!d_pin) init();
    gpio_write(d_pin, signal);
    return signal;
}

int pwm(unsigned int freq, unsigned int duty, unsigned int val){
    if(!d_pin)init();
    
    freq = freq==0 ? 1:freq;
    duty = duty%101;
    
    unsigned int us = 1000000; // work in micro secs (us)
    unsigned int T =  (us / freq);

    unsigned int on_us  = T * duty / 100.; // convert duty into percent
    unsigned int off_us = T - on_us;

    gpio_write(d_pin, val);
    delay_us(on_us); 
    gpio_write(d_pin, 0);
    delay_us(off_us);

    return 0;
}

unsigned int read(){
    if(!d_pin)init();
    return gpio_read(d_pin);
}

void beep_seconds(unsigned int freq, unsigned int duty, double duration_sec)
{
    if(freq == 0) return;

    unsigned int cycles = (unsigned int)(freq * duration_sec);

    for(unsigned int i = 0; i < cycles; i++)
    {
        pwm(freq, duty, 1);
    }
}

In [4]:
# sound on
write(1)
# expect 1 for on
read()

1

In [5]:
# sound off
write(0)
# expect 0 for off
read()

0

In [6]:
# test microblaze function
pwm(1,50,1)

0

In [7]:
import time

def measureTimeOfFn(fn,*args):
    tic = time.perf_counter()
    fn(*args)
    toc  = time.perf_counter()
    print("Elapsed time:", toc - tic, "seconds")
def pwmTest(testDurationInSec=1, hz=1, duty=1, val=1, debug=True): # in quiet mode I'm in public :)
    for i in range(testDurationInSec*hz): # forces beep rate for second(s)
        pwm(hz,duty,val)
        if debug: print(i)

In [ ]:
tic = time.perf_counter()
pwm(5,50,1)
toc  = time.perf_counter()
print("NO FN Elapsed time:", toc - tic, "seconds")

In [ ]:

durationInSec = 1
hz=10; duty=50; signal=1;
measureTimeOfFn(pwmTest, durationInSec, hz, duty, signal)

In [8]:
# headless test to vary duty cycle (volume)
# Choose 10Hz for testing
# sweep duty cycle from 0 to 100

# duty cycle test. Buzzer will slowly increase in volume. 
def testDutyCycle(hz=10, numTest=1):
    testIterations=0
    while testIterations < numTest:
        for i in range(0,101):
            pwm(hz,i,1)
            print("duty: %d%%"%i)
        testIterations = testIterations+1
    print("duty cycle test done.")
def sweepDutyCycle(hz=10, duty=101, duration=1, startDuty=0, step=1, debug=True):
    for i in range(startDuty, duty, step):
        pwmTest(testDurationInSec=duration, hz=hz, duty=i, val=1, debug=False)
        if debug: print("duty: %d%%"%i)
    if debug: print("duty cycle test done.")

In [ ]:
sweepDutyCycle(hz=30,duty=41,startDuty=20,step=10)

In [ ]:
testDutyCycle()

In [9]:
# headless test to vary frequency (tone)
# Freq test. Buzzer will slowly increase in tone. 
def testBuzzerFreq(hz=100, duty=1, duration=1, startHz=0, step=1, debug=True):
    for i in range(startHz, hz+1, step):
        pwmTest(testDurationInSec=duration, hz=i, duty=duty, val=1, debug=False)
        if debug: print("freq: %dHz"%i)
    if debug: print("Freq test done.")
def sweepFreq(hz=100, duty=1, duration=1, startHz=0, step=1, debug=True):
    testBuzzerFreq(hz=hz, duty=duty, duration=duration, startHz=startHz, step=step, debug=debug)

In [43]:
sweepFreq(hz=6,startHz=3,step=3,duty=1)

freq: 3Hz
freq: 6Hz
Freq test done.


In [ ]:
testBuzzerFreq()

In [ ]:
# make button listener
import threading
import time
def beep(durationInSec=5, hz=5, duty=50, val=1):
    pwmTest(testDurationInSec=durationInSec,hz=hz,duty=duty,val=val,debug=False)
    
def btnListener(btns, listenEvent, buzzEvent, closeEvent):
    print("btn listener active.")
    
    debounce=.5
    while(not closeEvent.is_set()):
        state = btns.read()
        
        if state & 0b0001: # btn 1
            print("btn 1 pressed")
            
            time.sleep(debounce)
        if state & 0b0010: # btn 2
            print("btn 2 pressed")
            time.sleep(debounce)
        if state & 0b0100: # btn 3
            print("btn 3 pressed")
            print("beeping...")
            beep()
            print("beeping done.")
            time.sleep(debounce)
        if state & 0b1000: # btn 4
            print("btn 4 pressed")
            time.sleep(debounce)
            break;
        
        time.sleep(.01)
        
    print("btn listener done.")
listenEvent = threading.Event()
buzzEvent = threading.Event()
closeEvent = threading.Event()

btnListenerThread = threading.Thread(target=btnListener,args=(btns, listenEvent, buzzEvent, closeEvent))
btnListenerThread.start()

In [ ]:
def siren():
    sweepFreq(hz=25, duty=50, duration=1,startHz=10, step=5, debug=False)
    #sweepDutyCycle()

In [ ]:
siren()

In [17]:
import threading
import time
import socket

def beep(durationInSec=5, hz=5, duty=50, val=1):
    pwmTest(testDurationInSec=durationInSec,hz=hz,duty=duty,val=val,debug=False)

def start_server(closeEvent, serverEvent, host="127.0.0.1", port=12345):
    if serverEvent.is_set():
        print("Server already running")
        return
    serverEvent.set()

    server = None
    conn = None
    
    try:
        server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind((host, port))
        server.listen(1)
        
        print("Listening on {}:{}".format(host,port))
        
        conn, addr = server.accept()
        conn.settimeout(1)

        print('Connected by', addr)

        while not closeEvent.is_set():
            try:
                data = conn.recv(1024)  
                if not data:
                    print("Client disconnected")
                    break

                msg = data.decode(errors="replace")

                if "beep" in msg:
                    beep(durationInSec=1,duty=1)
                else:
                    print("RX:", msg)
            except Exception as e:
                print("SERVER LOOP ERR: ",e)

        conn.close()
        server.close()
        
        print("SERVER THREAD DONE.")

    except Exception as e:
        print("Server error: ", e)
    finally:
        serverEvent.clear()
        
        
def client(closeEvent, clientEvent, sendEvent, server_ip="127.0.0.1", port=12345):
    if clientEvent.is_set():
        print("Client already running")
        return
    clientEvent.set()
    
    print("closeE: {} clientE: {}".format(closeEvent.is_set(),clientEvent.is_set()))
    s = None
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        s.connect((server_ip, port))
        
        print("Connected to {} on port {}".format(server_ip,port))
        while not closeEvent.is_set() and clientEvent.is_set():
            
            if sendEvent.is_set():
                s.sendall(b"BUZZ")
                sendEvent.clear()
            else:
                pass
            
        s.close()    
        print("CLIENT THREAD DONE.")
    except Exception as e:
        print("Client error: ", e)
    finally:
        clientEvent.clear()

def btnListener(btns, listenEvent, clientEvent, closeEvent):
    print("btn listener active.")
    
    debounce=.5
    sendEvent = threading.Event()
    
    while(not closeEvent.is_set()):
        state = btns.read()
        
        if state & 0b0001: # btn 1
            print("btn 1 pressed")
            if not listenEvent.is_set():
                threading.Thread(target=start_server, args=(closeEvent,listenEvent,)).start()
                
            time.sleep(debounce)
        if state & 0b0010: # btn 2
            print("btn 2 pressed")
            sendEvent.set()
            
            if not clientEvent.is_set():
                threading.Thread(target=client, args=(closeEvent,clientEvent, sendEvent, '192.168.2.99', 50007,)).start()
            time.sleep(debounce)
        
        if state & 0b0100: # btn 3
            #local beep test
            print("btn 3 pressed")
            print("beeping...")
            beep(duty=5)
            print("beeping done.")
            time.sleep(debounce)
        if state & 0b1000: # btn 4
            print("btn 4 pressed")
            closeEvent.set()
            sendEvent.clear()
            time.sleep(debounce)
            closeEvent.clear()
            break;
        
        time.sleep(.01)
        
    print("btn listener done.")

listenEvent = threading.Event()
buzzEvent = threading.Event()
closeEvent = threading.Event()


btnListenerThread = threading.Thread(target=btnListener,args=(btns, listenEvent, buzzEvent, closeEvent))
btnListenerThread.start()

btn listener active.
btn 2 pressed
closeE: False clientE: True
Client error:  [Errno 111] Connection refused
btn 2 pressed
closeE: False clientE: True
Client error:  [Errno 111] Connection refused
btn 2 pressed
closeE: False clientE: True
Client error:  [Errno 111] Connection refused
btn 2 pressed
closeE: False clientE: True
Client error:  [Errno 111] Connection refused
btn 2 pressed
closeE: False clientE: True
Connected to 192.168.2.99 on port 50007
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 2 pressed
btn 4 pressed
CLIENT THREAD DONE.
btn listener done.
